# 🎵 HelloSindh Music - Cloud Scraper (100% Free on Google Colab)

Yeh notebook Google ke high-speed cloud network (1 Gbps+) par chalegi.
Tamam 1,113 Sindhi songs 15–20 minutes mein download aur fingerprint ho kar ready ho jayenge.
Completed database automatically aapki **Google Drive** mein save ho jayega!

### Step 1: Google Drive Mount Karein (Optional lekin Recommended)
Is cell ko run karein taake output database seedha aapki Google Drive mein save ho jaye.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive successfully connected!")

### Step 2: `colab_files.zip` Load Karein
Agar aapne Google Drive ke main folder mein `colab_files.zip` rakhi hai toh yeh automatic wahan se utha lega. Warna "Choose Files" button se upload kar sakte hain.

In [ ]:
import os
import shutil
from google.colab import files

# Check if file exists in Google Drive
drive_path = '/content/drive/MyDrive/colab_files.zip'
if os.path.exists(drive_path):
    print("Found colab_files.zip in Google Drive! Copying...")
    shutil.copy(drive_path, 'colab_files.zip')
elif not os.path.exists('colab_files.zip'):
    print('Neeche "Choose Files" button par click karke colab_files.zip select karein:\n')
    uploaded = files.upload()

# Files uncompress karein
!unzip -o colab_files.zip
!chmod +x dataset_scraper_linux
print("\n[SUCCESS] Files ready for scraping!")

### Step 3: MariaDB (MySQL), yt-dlp & FFmpeg Setup Karein

In [ ]:
# MariaDB aur tools install karein
!apt-get update -qq && apt-get install -y mariadb-server ffmpeg -qq
!pip install -U yt-dlp -q
!service mariadb start

# Database aur tables initialize karein
!mysql -e "CREATE DATABASE IF NOT EXISTS shazam CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci;"
!mysql shazam < schema.sql
!mysql -e "SHOW TABLES IN shazam;"
print("\nMySQL Server & Database ready!")

### Step 4: High-Speed 16-Worker Cloud Ingestion Shuru Karein
Yeh cell cloud ke 1 Gbps Google network se 16 workers parallel chalayega.

In [ ]:
# Run high-speed scraper
!./dataset_scraper_linux -dataset sindhi_songs_dataset.json -workers 16 -limit 0

### Step 5: Progress Check Karein

In [ ]:
!mysql -e "SELECT count(*) AS total_songs FROM shazam.songs; SELECT id, title, artist, created_at FROM shazam.songs ORDER BY id DESC LIMIT 5;"

### Step 6: Completed Database Google Drive Mein Save & Download Karein
Jab tamam songs complete ho jayein, yeh cell run karein. File automatic aapki **Google Drive** mein save ho jayegi aur browser se download bhi shuru ho jayegi.

In [ ]:
# Export database dump
!mysqldump -u root --single-transaction --quick --max_allowed_packet=512M shazam > shazam_completed_1113.sql

# Zip for fast download
!zip -9 shazam_completed_1113.zip shazam_completed_1113.sql

# Save directly to Google Drive
import shutil
if os.path.exists('/content/drive/MyDrive'):
    shutil.copy('shazam_completed_1113.zip', '/content/drive/MyDrive/shazam_completed_1113.zip')
    print("\n[SUCCESS] File aapki Google Drive mein save ho gayi: 'shazam_completed_1113.zip'")

# Also trigger browser download
from google.colab import files
files.download('shazam_completed_1113.zip')
print("Browser download started!")